# 8. Route and assemble

Route each paper to one of the three DeltaBay questions, then assemble the full intermediate-debate JSON from upstream parquet/JSON artifacts.

Inputs (from `output/<RUN_ID>/`):
- `papers.parquet` (notebook 1)
- `resolved_posts.parquet` (notebook 3)
- `classified.parquet` (notebook 5, optionally appended by notebook 6)
- `clusters.json` (notebook 7)

Outputs: `output/<RUN_ID>/debate.json`; appends `stage_8_route_and_assemble` to `manifest.json`.

See `PIPELINE_PLAN.md` § S25 / S15 / S16 / S17.

## Setup

Pick the latest `RUN_ID` that has a `classified.parquet`, load `.env`.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = Path.cwd().parent.resolve()
OUTPUT_ROOT = REPO_ROOT / "altendor" / "output"

load_dotenv(REPO_ROOT / ".env")

# Override RUN_ID here if you want a specific historical run instead of the latest.
RUN_ID: str | None = None
if RUN_ID is None:
    candidates = sorted([p.name for p in OUTPUT_ROOT.iterdir() if (p / "classified.parquet").exists()])
    if not candidates:
        raise FileNotFoundError("No run with classified.parquet found. Run notebook 5 first.")
    RUN_ID = candidates[-1]

OUT_DIR = OUTPUT_ROOT / RUN_ID
print(f"Using run id: {RUN_ID}\nOutput dir: {OUT_DIR}")

In [ ]:
import json
import pandas as pd

papers_df = pd.read_parquet(OUT_DIR / "papers.parquet")
resolved_df = pd.read_parquet(OUT_DIR / "resolved_posts.parquet")
classified_df = pd.read_parquet(OUT_DIR / "classified.parquet")
clusters_raw = json.loads((OUT_DIR / "clusters.json").read_text()) if (OUT_DIR / "clusters.json").exists() else {}
print(f"papers={len(papers_df)} resolved={len(resolved_df)} classified={len(classified_df)} clusters_keys={len(clusters_raw)}")

## Route papers to Questions

One Claude call per paper picks the best of the three DeltaBay questions with a confidence score; `diversify_routes` then balances the assignment so every question gets at least one paper when possible.

In [ ]:
import anthropic
from altendor.route.question_router import (
    PaperForRouting, diversify_routes, route_paper_to_question,
)

client = anthropic.Anthropic()
raw_routes: dict[str, tuple[str, float]] = {}
for _, p in papers_df.iterrows():
    if not p.get("doi"):
        continue
    pp = PaperForRouting(doi=str(p["doi"]), title=str(p["title"]), abstract=(p.get("abstract") or None))
    try:
        qid, conf = route_paper_to_question(client, pp)
        raw_routes[pp.doi] = (qid, float(conf))
    except Exception as exc:
        print(f"  route failed for {pp.doi}: {exc}")

final_routes = diversify_routes(raw_routes)
from collections import Counter
route_dist = Counter(final_routes.values())
print("Final route distribution:")
for qid, n in route_dist.items():
    print(f"  {qid}: {n}")

## Reconstruct upstream dataclasses for the builder

Parquet round-trips lose the upstream dataclass types; rehydrate `ResolvedPost`, the classification union (`Endorsement` / `Flag` / `Irrelevant`), and `ClaimCluster` so the builder can consume them.

In [ ]:
from altendor.cluster.claims import ClaimCluster
from altendor.classify.schema import Endorsement, Flag, Irrelevant
from altendor.enrich.text_resolver import ResolvedPost

resolved_posts: dict[str, ResolvedPost] = {}
for _, r in resolved_df.iterrows():
    resolved_posts[r["post_id"]] = ResolvedPost(
        post_id=r["post_id"], platform=r["platform"], text=r["text"],
        author_handle=r.get("author_handle"), author_id=r.get("author_id"),
        url=r["url"], created_at=r["created_at"],
        raw_title=r["raw_title"], text_confidence=r["text_confidence"],
    )

classified: dict[str, object] = {}
for _, r in classified_df.iterrows():
    k = r["kind"]
    if k == "endorsement":
        classified[r["post_id"]] = Endorsement(
            claim_text=r["claim_text"] or "",
            magnitude_dB=int(r["magnitude_dB"]) if pd.notna(r["magnitude_dB"]) else 1,
            criterion=r["criterion"] or "Support",
            reasoning=r["reasoning"] or "",
        )
    elif k == "flag":
        classified[r["post_id"]] = Flag(category=r["category"] or "other", rationale=r["rationale"] or "")
    elif k == "irrelevant":
        classified[r["post_id"]] = Irrelevant(reason=r["reason"] or "")

clusters: dict[str, list[ClaimCluster]] = {}
for ro_id, cs in clusters_raw.items():
    clusters[ro_id] = [ClaimCluster(canonical_text=c["canonical_text"], member_post_ids=list(c["member_post_ids"])) for c in cs]

print(f"resolved_posts={len(resolved_posts)} classified={len(classified)} clusters keys={len(clusters)}")

## Assemble + write

`build_intermediate` stitches the routes, clusters, classifications, and resolved posts into an `IntermediateDebate`; `write_debate_json` serialises it to disk.

In [ ]:
from altendor.assemble.builder import build_intermediate
from altendor.assemble.deltabay_writer import write_debate_json

idb = build_intermediate(
    papers=papers_df,
    resolved_posts=resolved_posts,
    classified=classified,
    clusters=clusters,
    routes=final_routes,
    run_id=RUN_ID,
)

out_path = OUT_DIR / "debate.json"
write_debate_json(idb, out_path)
print(f"Wrote {out_path}")

## Summary table (S31)

One row per paper with the routed question, number of subclaims, and the endorsement / flag counts (paper-level evidence + per-subclaim).

In [ ]:
rows = []
for p in idb.papers:
    n_end = sum(len(s.endorsements) for s in p.answer.subclaims) + len(p.answer.evidence.endorsements)
    n_flag = len(p.answer.evidence.flags) + sum(len(s.flags) for s in p.answer.subclaims)
    rows.append({
        "doi": p.doi,
        "routed_question": p.routed_question_id,
        "subclaims": len(p.answer.subclaims),
        "endorsements": n_end,
        "flags": n_flag,
    })
summary_df = pd.DataFrame(rows)
print(summary_df.to_markdown(index=False) if hasattr(summary_df, "to_markdown") else summary_df.to_string(index=False))
print(f"\nTotal participants: {len(idb.participants)}")

In [ ]:
manifest_path = OUT_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
manifest["stage_8_route_and_assemble"] = {
    "n_papers": len(idb.papers),
    "n_participants": len(idb.participants),
    "route_distribution": {qid: int(n) for qid, n in route_dist.items()},
}
manifest.setdefault("output_files", []).append(str(out_path.relative_to(REPO_ROOT)))
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True))
print(f"Updated {manifest_path}")